In [0]:
bucket_path = "s3a://capgemini-retail-etl-shiva/raw/"

files = dbutils.fs.ls(bucket_path)

for file in files:
    print(file.name)

In [0]:
# Define file types to process with their archive folders
file_types = [
    {"prefix": "customers_src", "archive_folder": "customers"},
    {"prefix": "products_src", "archive_folder": "products"},
    {"prefix": "sales_transactions_src", "archive_folder": "sales_transactions"},
    {"prefix": "stores_src", "archive_folder": "stores"}
]

print(f"Processing {len(file_types)} file types:")
for ft in file_types:
    print(f"  - {ft['prefix']} → archive/{ft['archive_folder']}/")

In [0]:
# Initialize archival log
archival_log = []

# Process each file type
for file_type in file_types:
    prefix = file_type["prefix"]
    archive_folder = file_type["archive_folder"]
    
    print(f"\n{'='*60}")
    print(f"Processing: {prefix}")
    print(f"{'='*60}")
    
    # Filter files by prefix
    matching_files = [f.path for f in files if f.name.startswith(prefix)]
    
    if len(matching_files) == 0:
        print(f"⚠️  No files found with prefix '{prefix}'")
        continue
    
    print(f"Found {len(matching_files)} file(s)")
    
    # Sort files (filename contains timestamp, so sorting works)
    matching_files_sorted = sorted(matching_files)
    
    # Get latest and old files
    latest_file = matching_files_sorted[-1]
    old_files = matching_files_sorted[:-1]
    
    latest_filename = latest_file.split("/")[-1]
    print(f"✓ Latest file: {latest_filename}")
    
    if len(old_files) == 0:
        print(f"✓ No old files to archive")
        continue
    
    print(f"📦 Archiving {len(old_files)} old file(s)...")
    
    # Archive path with subfolder per file type
    archive_path = f"s3a://capgemini-retail-etl-shiva/archive/{archive_folder}/"
    
    # Archive old files
    for file_path in old_files:
        file_name = file_path.split("/")[-1]
        destination = archive_path + file_name
        
        try:
            dbutils.fs.mv(file_path, destination)
            print(f"  ✓ Moved: {file_name}")
            
            # Log archival
            archival_log.append({
                "FileType": prefix,
                "FileName": file_name,
                "Source": file_path,
                "Destination": destination,
                "Status": "Archived"
            })
        except Exception as e:
            print(f"  ✗ Failed to move {file_name}: {str(e)}")
            archival_log.append({
                "FileType": prefix,
                "FileName": file_name,
                "Source": file_path,
                "Destination": destination,
                "Status": f"Failed: {str(e)}"
            })

print(f"\n{'='*60}")
print(f"Archival Complete: {len(archival_log)} file(s) processed")
print(f"{'='*60}")

In [0]:
# Display archival log
if len(archival_log) > 0:
    print("\n📋 Archival Log:")
    print("="*80)
    display(archival_log)
else:
    print("ℹ️ No files were archived in this run")

In [0]:
# Verify remaining files in raw bucket
print("🔍 Remaining files in raw bucket:")
print("="*60)

remaining_files = dbutils.fs.ls(bucket_path)

if len(remaining_files) == 0:
    print("⚠️  No files found in raw bucket")
else:
    for file in remaining_files:
        print(f"  ✓ {file.name}")